In [2]:
import fastapi
import uvicorn
import sklearn
import pandas
import joblib
import requests

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("All required packages imported successfully!")

FastAPI: 0.115.8
Uvicorn: 0.34.0
Scikit-learn: 1.6.1
Pandas: 2.2.3
All required packages imported successfully!


In [3]:
!pip install -q fastapi==0.115.8 uvicorn==0.34.0 scikit-learn==1.6.1 pandas==2.2.3 joblib==1.4.2 httpx==0.28.1

In [4]:
from google.colab import files

uploaded = files.upload()

Saving main.py to main.py
Saving schemas.py to schemas.py


In [5]:
from google.colab import files

uploaded = files.upload()

Saving champion_model.joblib to champion_model.joblib


In [6]:
import os
import shutil

os.makedirs("app", exist_ok=True)
os.makedirs("models", exist_ok=True)

if os.path.exists("main.py"):
    shutil.move("main.py", "app/main.py")

if os.path.exists("schemas.py"):
    shutil.move("schemas.py", "app/schemas.py")

if os.path.exists("champion_model.joblib"):
    shutil.move(
        "champion_model.joblib",
        "models/champion_model.joblib"
    )

open("app/__init__.py", "w").close()

print("Files organized successfully!")

Files organized successfully!


In [7]:
!ls -R

.:
app  models  sample_data

./app:
__init__.py  main.py  schemas.py

./models:
champion_model.joblib

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [8]:

import threading
import uvicorn

config = uvicorn.Config(
    "app.main:app",
    host="127.0.0.1",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

print("FastAPI server is starting...")


FastAPI server is starting...


In [9]:

import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status Code:", response.status_code)
print("Response:", response.json())


INFO:     127.0.0.1:52716 - "GET / HTTP/1.1" 200 OK
Status Code: 200
Response: {'message': 'Titanic Survival Prediction API is running'}


In [10]:

response = requests.get("http://127.0.0.1:8000/health")

print("Status Code:", response.status_code)
print("Response:", response.json())


INFO:     127.0.0.1:55492 - "GET /health HTTP/1.1" 200 OK
Status Code: 200
Response: {'status': 'healthy', 'model_loaded': True}


In [11]:

sample_passenger = {
    "pclass": 1,
    "sex": "female",
    "age": 29,
    "sibsp": 0,
    "parch": 0,
    "fare": 100.0,
    "embarked": "S",
    "who": "woman",
    "adult_male": False,
    "deck": "C",
    "embark_town": "Cherbourg",
    "alone": True
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_passenger
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())


INFO:     127.0.0.1:56080 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Prediction: {'prediction': 1, 'prediction_label': 'Survived', 'probability': 0.9419368803364608}


In [12]:
!mkdir -p tests

In [13]:
%%writefile tests/test_api.py

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)


def test_home():
    response = client.get("/")
    assert response.status_code == 200


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_predict():
    passenger = {
        "pclass": 1,
        "sex": "female",
        "age": 29,
        "sibsp": 0,
        "parch": 0,
        "fare": 100.0,
        "embarked": "S",
        "who": "woman",
        "adult_male": False,
        "deck": "C",
        "embark_town": "Cherbourg",
        "alone": True
    }

    response = client.post("/predict", json=passenger)

    assert response.status_code == 200
    assert response.json()["prediction"] in [0, 1]
    assert 0 <= response.json()["probability"] <= 1


def test_invalid_input():
    passenger = {
        "pclass": 4,
        "sex": "female",
        "age": 29,
        "sibsp": 0,
        "parch": 0,
        "fare": 100.0,
        "embarked": "S",
        "who": "woman",
        "adult_male": False,
        "deck": "C",
        "embark_town": "Cherbourg",
        "alone": True
    }

    response = client.post("/predict", json=passenger)

    assert response.status_code == 422

Writing tests/test_api.py


In [14]:
!pytest -v tests/test_api.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.15, typeguard-4.6.0, platformdirs-4.14.0, anyio-4.15.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
ImportError while importing test module '/content/tests/test_api.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_api.py:3: in <module>
    from app.main import app
E   ModuleNotFoundError: No module named 'app'
=========================

In [15]:
!pwd
!ls -R /content

/content
/content:
app  models  sample_data  tests

/content/app:
__init__.py  main.py  __pycache__  schemas.py

/content/app/__pycache__:
__init__.cpython-313.pyc  main.cpython-313.pyc	schemas.cpython-313.pyc

/content/models:
champion_model.joblib

/content/sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

/content/tests:
__pycache__  test_api.py

/content/tests/__pycache__:
test_api.cpython-313-pytest-8.4.2.pyc


In [16]:
!touch /content/app/__init__.py

In [17]:
import sys
import os

sys.path.insert(0, "/content")

print("Project path added:", sys.path[0])
print("App folder exists:", os.path.exists("/content/app"))
print("Main file exists:", os.path.exists("/content/app/main.py"))

Project path added: /content
App folder exists: True
Main file exists: True


In [18]:
!python -m pytest -v /content/tests/test_api.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.15, typeguard-4.6.0, platformdirs-4.14.0, anyio-4.15.1
collected 4 items                                                              

tests/test_api.py::test_home PASSED                                      [ 25%]
tests/test_api.py::test_health PASSED                                    [ 50%]
tests/test_api.py::test_predict PASSED                                   [ 75%]
tests/test_api.py::test_invalid_input PASSED                             [100%]

============================== 4 passed in 1.70s ===============================
